# Module 15: Advanced OOP — Inheritance, Dunder Methods & More

Building on Module 14, this module covers inheritance, polymorphism, dunder methods, and OOP design principles.

## 15.1 Inheritance

Inheritance lets a **child class** reuse code from a **parent class**, adding or overriding behavior.

```python
class Child(Parent):
    ...
```

In [ ]:
class Animal:
    """Base class for all animals."""
    
    def __init__(self, name, sound):
        self.name  = name
        self.sound = sound
    
    def speak(self):
        return f"{self.name} says {self.sound}!"
    
    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class Dog(Animal):
    """Dog inherits from Animal."""
    
    def __init__(self, name, breed):
        super().__init__(name, "Woof")   # call parent __init__
        self.breed = breed
    
    def fetch(self, item):
        return f"{self.name} fetches the {item}!"


class Cat(Animal):
    def __init__(self, name, indoor=True):
        super().__init__(name, "Meow")
        self.indoor = indoor
    
    def purr(self):
        return f"{self.name} purrs..."


dog = Dog("Rex", "Labrador")
cat = Cat("Whiskers")

print(dog.speak())        # inherited from Animal
print(dog.fetch("ball"))  # Dog-specific
print(cat.speak())        # inherited, different sound
print(cat.purr())         # Cat-specific
print(repr(dog))          # uses Animal.__repr__

## 15.2 `super()` in Detail

`super()` gives you access to the parent class. Always call it in `__init__` when the parent has its own initialization:

In [ ]:
class Vehicle:
    def __init__(self, make, model, year):
        self.make  = make
        self.model = model
        self.year  = year
    
    def describe(self):
        return f"{self.year} {self.make} {self.model}"


class ElectricVehicle(Vehicle):
    def __init__(self, make, model, year, battery_kwh):
        super().__init__(make, model, year)   # init the parent
        self.battery_kwh = battery_kwh        # extra attribute
    
    def describe(self):
        base = super().describe()    # call parent's method
        return f"{base} (Electric, {self.battery_kwh}kWh)"


tesla = ElectricVehicle("Tesla", "Model 3", 2024, 82)
print(tesla.describe())
print(tesla.make)    # inherited attribute

## 15.3 Polymorphism

**Polymorphism** means different objects respond to the same method call in different ways. Python achieves this naturally — no special syntax needed:

In [ ]:
class Shape:
    def area(self):
        raise NotImplementedError("Subclasses must implement area()")
    
    def describe(self):
        return f"{type(self).__name__}: area = {self.area():.2f}"


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    def area(self):
        import math
        return math.pi * self.radius ** 2


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width  = width
        self.height = height
    
    def area(self):
        return self.width * self.height


class Triangle(Shape):
    def __init__(self, base, height):
        self.base   = base
        self.height = height
    
    def area(self):
        return 0.5 * self.base * self.height


# Polymorphism: same call, different behavior
shapes = [Circle(5), Rectangle(4, 6), Triangle(3, 8)]
for shape in shapes:
    print(shape.describe())   # each calls its own area()

## 15.4 Dunder (Magic) Methods

Dunder methods (`__name__`) let your objects integrate with Python's built-in operations like `+`, `len()`, comparisons, and `print()`.

In [ ]:
class Vector:
    """2D vector with operator overloading."""
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"
    
    def __str__(self):
        return f"({self.x}, {self.y})"
    
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)
    
    def __len__(self):
        return 2   # a 2D vector has 2 components
    
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    
    def magnitude(self):
        return (self.x**2 + self.y**2) ** 0.5


a = Vector(1, 2)
b = Vector(3, 4)

print(a + b)         # uses __add__   → (4, 6)
print(a - b)         # uses __sub__   → (-2, -2)
print(a * 3)         # uses __mul__   → (3, 6)
print(len(a))        # uses __len__   → 2
print(a == Vector(1,2))  # uses __eq__ → True
print(str(a))        # uses __str__
print(repr(a))       # uses __repr__
print(b.magnitude()) # 5.0

In [ ]:
# More useful dunder methods:
class SmartList:
    """A list wrapper with extra features."""
    
    def __init__(self, *items):
        self._data = list(items)
    
    def __len__(self):
        return len(self._data)
    
    def __getitem__(self, index):
        return self._data[index]
    
    def __setitem__(self, index, value):
        self._data[index] = value
    
    def __contains__(self, item):
        return item in self._data
    
    def __iter__(self):
        return iter(self._data)
    
    def __repr__(self):
        return f"SmartList{tuple(self._data)}"


sl = SmartList(10, 20, 30, 40)
print(len(sl))           # __len__
print(sl[1])             # __getitem__
print(20 in sl)          # __contains__
for item in sl:          # __iter__
    print(item, end=" ")
print()

## 15.5 Abstract Base Classes

To force subclasses to implement specific methods, use `ABC`:

In [ ]:
from abc import ABC, abstractmethod

class PaymentProcessor(ABC):
    """Abstract base — cannot be instantiated directly."""
    
    @abstractmethod
    def process_payment(self, amount):
        """All subclasses MUST implement this."""
        ...
    
    @abstractmethod
    def refund(self, amount):
        ...
    
    def validate_amount(self, amount):
        """Concrete method — shared by all subclasses."""
        if amount <= 0:
            raise ValueError("Amount must be positive")


class StripeProcessor(PaymentProcessor):
    def process_payment(self, amount):
        self.validate_amount(amount)
        return f"Stripe: charged ${amount:.2f}"
    
    def refund(self, amount):
        self.validate_amount(amount)
        return f"Stripe: refunded ${amount:.2f}"


# This would raise TypeError:
# p = PaymentProcessor()  # Cannot instantiate abstract class

stripe = StripeProcessor()
print(stripe.process_payment(99.99))
print(stripe.refund(9.99))

## ✅ Module 15 Summary

| Concept | Key point |
|---------|----------|
| Inheritance | `class Child(Parent):` |
| `super()` | Call parent methods |
| Override | Redefine a method in the child |
| Polymorphism | Same interface, different behavior |
| Dunder methods | `__repr__`, `__add__`, `__len__`, etc. |
| ABC | Force subclass to implement methods |

---
### 🏋️ Practice!
Open **`exercises/ex_15_advanced_oop.py`**